# 문항 1 네이버 VIBE Top 100 수집

In [ ]:
import requests
import pandas as pd
import json

URL = "https://apis.naver.com/vibeWeb/musicapiweb/vibe/v1/chart/track/total"
PARAMS = {
    "start" : "1",
    "display" : "100",
}
HEADERS = {
    # "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    # "Origin": "https://vibe.naver.com",
    "Accept" : "application/json",
}

songs = []

response = requests.get(URL, params=PARAMS, headers=HEADERS).json()

result = response["response"]["result"]["chart"]["items"]["tracks"]
for song in result:
    artists = [artist["artistName"] for artist in song["artists"]]
    songs.append({
        "순위": song["rank"]["currentRank"],
        "곡명": song["trackTitle"],
        "아티스트": artists,
    })

df = pd.DataFrame(songs)
print("건수: " + str(len(df)))
df.to_csv("vibe_top100.csv", index=False, encoding="utf-8-sig")

건수: 100


# 문항 2 삼성전자 일별 시세 1년치 수집

In [91]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re

URL = "https://finance.naver.com/item/sise_day.naver"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",

}


result = []
count = 0
# 365일 이므로 38 사용
for i in range(1, 38):
    # 1년치 데이터 수집후 멈추기.
    if count >= 365:
        break

    PARAMS = {
        "code" : "005930",
        "page" : i
    }

    response = requests.get(URL, params=PARAMS, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")
    trs = soup.select('table.type2 tr[onmouseover="mouseOver(this)"]')
    for tr in trs:
        # 1년치 데이터 수집후 멈추기.
        if count >= 365:
            break
        
        change = tr.select_one("span.blind").text.strip()
        amount = tr.select("td")[2].text.strip()
        numeric_amount = re.sub(r'[\D]', '', amount)

        new_change = change + " " + numeric_amount
        result.append({
            "날짜" : tr.select_one("td:nth-child(1)").text.strip(),
            "종가" : int(tr.select_one("td:nth-child(2)").text.strip().replace(",", "")),
            "전일비" : new_change,
            "시가" : int(tr.select_one("td:nth-child(4)").text.strip().replace(",", "")),
            "고가" : int(tr.select_one("td:nth-child(5)").text.strip().replace(",", "")),
            "저가" : int(tr.select_one("td:nth-child(6)").text.strip().replace(",", "")),
            "거래량" : int(tr.select_one("td:nth-child(7)").text.strip().replace(",", "")),
        })
        count += 1

df = pd.DataFrame(result)
df.to_csv("samsung_1y.csv", index=False, encoding="utf-8-sig")

# 문항 3 네이버 뉴스 검색기 함수 만들기 (스크롤 포함) - 난이도 상

In [114]:
payload = "abt=null&cluster_rank=110&de=&ds=&eid=&field=0&force_original=&is_dts=0&is_sug_officeid=0&mynews=0&news_office_checked=&nlu_query=&nqx_theme=%7B%22theme%22%3A%7B%22sub%22%3A%5B%7B%22name%22%3A%22society%22%7D%2C%7B%22name%22%3A%22weather%22%7D%5D%7D%7D&nso=so%3Ar%2Cp%3Aall%2Ca%3Aall&nx_and_query=&nx_search_hlquery=&nx_search_query=&nx_sub_query=&office_category=&office_section_code=0&office_type=0&pd=0&photo=0&qdt=0&query=%EA%B2%BD%EC%A0%9C&query_original=&rev=0&service_area=&sm=tab_smr&sort=0&spq=2&ssc=tab.news.all&start=51"

payload = [p.split('=') for p in payload.split('&')]
payload = {k: v for k, v in payload}

payload

{'abt': 'null',
 'cluster_rank': '110',
 'de': '',
 'ds': '',
 'eid': '',
 'field': '0',
 'force_original': '',
 'is_dts': '0',
 'is_sug_officeid': '0',
 'mynews': '0',
 'news_office_checked': '',
 'nlu_query': '',
 'nqx_theme': '%7B%22theme%22%3A%7B%22sub%22%3A%5B%7B%22name%22%3A%22society%22%7D%2C%7B%22name%22%3A%22weather%22%7D%5D%7D%7D',
 'nso': 'so%3Ar%2Cp%3Aall%2Ca%3Aall',
 'nx_and_query': '',
 'nx_search_hlquery': '',
 'nx_search_query': '',
 'nx_sub_query': '',
 'office_category': '',
 'office_section_code': '0',
 'office_type': '0',
 'pd': '0',
 'photo': '0',
 'qdt': '0',
 'query': '%EA%B2%BD%EC%A0%9C',
 'query_original': '',
 'rev': '0',
 'service_area': '',
 'sm': 'tab_smr',
 'sort': '0',
 'spq': '2',
 'ssc': 'tab.news.all',
 'start': '51'}

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

def crawl_naver_news(keyword, page):
    URL = "https://s.search.naver.com/p/newssearch/3/api/tab/more"
    HEADERS = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    }

    news = []

    for i in range(1, page + 1):
        PAYLOAD = {
            'ssc': 'tab.news.all',
            'query': keyword,
            'start': str(((i - 1) * 10) + 1),
        }

        response = requests.get(URL, params=PAYLOAD, headers=HEADERS)
        if response.status_code == 200:
            print(f"Page: {i}, 200: Successfully crawled")
        else:
            return f"Error: {response.status_code}"

        data = response.json()["collection"][0]["html"]
        soup = BeautifulSoup(data, "html.parser")
        news_data = soup.select("div.fds-news-item-list-tab > div")
        for item in news_data:
            news.append({
                "제목" : item.select_one("span.sds-comps-text-type-headline1").text.strip(),
                "언론사" : item.select_one("span.sds-comps-text-weight-sm span:nth-of-type(1)").text.strip(),
                "링크" : item.select_one("a.fender-ui_228e3bd1.ise_N8_wf_ooMaN2").get("href"),
                "요약" : item.select_one("span.sds-comps-text-type-body1").text.strip(),
            })

    df = pd.DataFrame(news)
    df.to_csv(f"news_{keyword}.csv", index=False, encoding="utf-8-sig")

crawl_naver_news("경제", 2)
crawl_naver_news("AI", 10)

Page: 1, 200: Successfully crawled
Page: 2, 200: Successfully crawled
Page: 1, 200: Successfully crawled
Page: 2, 200: Successfully crawled
Page: 3, 200: Successfully crawled
Page: 4, 200: Successfully crawled
Page: 5, 200: Successfully crawled
Page: 6, 200: Successfully crawled
Page: 7, 200: Successfully crawled
Page: 8, 200: Successfully crawled
Page: 9, 200: Successfully crawled
Page: 10, 200: Successfully crawled
